In [1]:
import os
import subprocess

def run_command(cmd):
    result = subprocess.run(
        cmd,
        shell=True,
        capture_output=True,
        text=True
    )
    return result.stdout.strip()


cpu_model = run_command("lscpu | grep 'Model name' | cut -d: -f2")
cores = run_command("lscpu | grep 'Core(s) per socket' | cut -d: -f2")
sockets = run_command("lscpu | grep 'Socket(s)' | cut -d: -f2")
threads = run_command("lscpu | grep 'Thread(s) per core' | cut -d: -f2")
numa_nodes = run_command("lscpu | grep 'NUMA node(s)' | cut -d: -f2")
cache_size = run_command("lscpu | grep 'L3 cache' | cut -d: -f2")
vector_units = run_command("lscpu | grep '^CPU(s):' | cut -d: -f2")

print(f"CPU Model: {cpu_model}")
print(f"Cores per Socket: {cores}")
print(f"Sockets: {sockets}")
print(f"Threads per Core: {threads}")
print(f"NUMA Nodes: {numa_nodes}")
print(f"L3 Cache Size: {cache_size}")
print(f"Logical CPUs: {vector_units}")



CPU Model: Intel(R) Xeon(R) CPU @ 2.20GHz
Cores per Socket: 1
Sockets: 1
Threads per Core: 2
NUMA Nodes: 1
L3 Cache Size: 55 MiB (1 instance)
Logical CPUs: 2


In [2]:
from genericpath import exists
import os
#1 Create a simple execute shell script

with open('student_tutorial.sh','w') as f:
    f.write(''' #!/bin/bash
    echo "=== System Info ==="
    echo "User: $(whoami)"
    echo "Path: $(pwd)"
    ''')
os.chmod('student_tutorial.sh', 0o755)
!./student_tutorial.sh

#2. Generate SSH keys Simply

if not os.path.exists('/root/.ssh/id_rsa'):
    !mkdir -p ~/.ssh && ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N ""

#3. Create simple github Actions template

os.makedirs("./github/workflows",exist_ok=True)
with open('./github/workflows/ci.yml','w') as f:
    f.write(''' name: Test Script
  on: [push]
  jobs:
    run-script:
      runs on: ubuntu-latest
      steps:
      - uses: actions/checkout@v4
      - run: chmod +x ./student_tutorial.sh && ./student_tutorial.sh
    ''')
print("\n SetUp compelete! Simple files are ready")
print("=========================================")
print("Copy your PUBLIC SSH KEY BELOW")
print("=========================================")
with open('/root/.ssh/id_rsa.pub') as f:
    print(f.read().strip())
print("===========================================")

=== System Info ===
User: root
Path: /content
Generating public/private rsa key pair.
Your identification has been saved in /root/.ssh/id_rsa
Your public key has been saved in /root/.ssh/id_rsa.pub
The key fingerprint is:
SHA256:0BJdtDXa3aCV5Ety6VjVjTmsXyhuEWEcQv16h8vvKXY root@770efa360cb4
The key's randomart image is:
+---[RSA 4096]----+
|      .. +=o*=+++|
|       o.  B+**=o|
|      o . o ++Bo.|
|       o    +Bo..|
|        S  ..=oo |
|            + + .|
|           . o o |
|             ooE.|
|            . o+o|
+----[SHA256]-----+

 SetUp compelete! Simple files are ready
Copy your PUBLIC SSH KEY BELOW
ssh-rsa AAAAB3NzaC1yc2EAAAADAQABAAACAQC0WQxoNj0Ex3VlS32QSUvvJMTEwe4mKZ1qLasGPeAXaZjodcewKh2xdcHGNQDhljSnXn9XUjluBJ1aqlW3qPjJGkmhe915K0OmQAhT8L+oKAn6i7+Rbt32Vy/wHTE7VGD9Eqg25083pRTetaCoW2Y+A8zswCI729xNRDp993xjsTrgEc3unqwfNE4XiYlpnee//hMDsmYtXvAAKTNuy9IK8+1lbpAw6be5BIUDQnrhB5ivONsF/RUHh/1F6egyp9MFLLbb1p25oXxsT4gh8V16qab4/T9w2KvIeLjJoSEB90Kqh7QVMA4xiLFqqNmWrCPnmUZlocY63jh1EFsado+Pv6E7le

In [3]:
import os

# 1. Fill your GitHub details below:
GITHUB_USERNAME = "bhavyatan"
GITHUB_EMAIL = "bhavya.t145@gmail.com"
REPO_NAME = "hpc-training-ssh-setup"

print("⚙️ Configuring Git details...")

os.system(f'git config --global user.name "{GITHUB_USERNAME}"')
os.system(f'git config --global user.email "{GITHUB_EMAIL}"')

print("✓ Git credentials configured locally!")

print("\n📁 Initializing Git Repository locally...")
os.system('git init')
os.system('git branch -M main')

print("✓ Git Repository initialized.")

# Adding code to remote
os.system(
    f'git remote add origin '
    f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
)

# Add files to staging
os.system('git add .')

# Commit changes
os.system('git commit -m "Initial commit"')

# Push code to GitHub
os.system('git push -u origin main')

⚙️ Configuring Git details...
✓ Git credentials configured locally!

📁 Initializing Git Repository locally...
✓ Git Repository initialized.


32768

In [4]:
!ssh -o StrictHostKeyChecking=no -T git@github.com

git@github.com: Permission denied (publickey).


In [5]:
import os

print("[*] Checking for existing Intel oneAPI installation...")
has_oneapi = os.path.exists("/opt/intel/oneapi")

if not has_oneapi:
    print("[*] Configuring Intel Official Package Repository...")
    commands = '''
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    # Install standalone oneAPI DPC++/C++ compiler and MPI developer headers
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    '''
    # We execute setup in background / check readiness
    print("[+] Intel repository commands prepared.")
else:
    print("[+] Intel oneAPI is already installed at /opt/intel/oneapi!")

[*] Checking for existing Intel oneAPI installation...
[*] Configuring Intel Official Package Repository...
[+] Intel repository commands prepared.


In [6]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

[*] Installing Intel oneAPI packages (this might take a few minutes)... 
deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://apt.repos.intel.com/oneapi all InRelease [5,684 B]
Get:3 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Hit:4 http://archive.ubuntu.com/ubuntu noble InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:9 https://apt.repos.intel.com/oneapi all/main all Packages [270 kB]
Get:10 https://apt.repos.intel.com/oneapi all/main amd64 Packages [960 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:12 http://archive.ubunt

In [7]:
import subprocess

# Execute the prepared installation commands in the shell environment
if not has_oneapi:
    print("[*] Installing Intel oneAPI packages (this might take a few minutes)... ")
    process = subprocess.Popen(commands, shell=True, executable='/bin/bash', stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

    # Stream output in real-time
    for line in iter(process.stdout.readline, ''):
        print(line, end='')

    process.stdout.close()
    return_code = process.wait()
    if return_code == 0:
        print("\n[+] Intel oneAPI environment installed successfully! Please re-run the compilation cells.")
    else:
        print(f"\n[!] Installation failed with exit code: {return_code}")
else:
    print("[+] Intel oneAPI is already available.")

[*] Installing Intel oneAPI packages (this might take a few minutes)... 
deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main
Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 https://apt.repos.intel.com/oneapi all InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu noble InRelease
Hit:5 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:6 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease
Hit:7 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease
Reading package lists...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dep

In [8]:
import os
os.makedirs('/opt/intel/oneapi', exist_ok=True)

# Create a mock/redirecting setvars.sh that exports paths to icpx and system-installed oneAPI tools
setvars_content = """#!/bin/bash
# Automatically exported configurations for oneAPI components
export PATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin
export PATH=$PATH:$(dirname $(which icpx 2>/dev/null || echo '/usr/bin'))
echo ':: oneAPI environment initialized manually ::'
"""

with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
    f.write(setvars_content)

print('[+] Successfully generated /opt/intel/oneapi/setvars.sh')

[+] Successfully generated /opt/intel/oneapi/setvars.sh


In [9]:
!bash -c "source /opt/intel/oneapi/setvars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query"

:: oneAPI environment initialized manually ::
bash: line 1: icpx: command not found


In [10]:
import os
import subprocess

# Let's locate the actual vars.sh or icpx path under /opt/intel/oneapi
print("Searching for icpx or setvars in /opt/intel...")
paths = []
for root, dirs, files in os.walk("/opt/intel"):
    for file in files:
        if file in ["setvars.sh", "vars.sh", "icpx"]:
            paths.append(os.path.join(root, file))

print("Found matching files:")
for p in paths:
    print(p)

Searching for icpx or setvars in /opt/intel...
Found matching files:
/opt/intel/oneapi/setvars.sh
/opt/intel/oneapi/mpi/2021.19/etc/mpi/vars.sh
/opt/intel/oneapi/mpi/2021.19/env/vars.sh
/opt/intel/oneapi/libfabric/2.6/etc/libfabric/vars.sh
/opt/intel/oneapi/libfabric/2.6/env/vars.sh
/opt/intel/oneapi/umf/1.2/env/vars.sh
/opt/intel/oneapi/compiler/2026.2/bin/icpx
/opt/intel/oneapi/compiler/2026.2/etc/compiler/vars.sh
/opt/intel/oneapi/compiler/2026.2/env/vars.sh
/opt/intel/oneapi/tcm/1.6/env/vars.sh
/opt/intel/oneapi/debugger/2026.2/etc/debugger/vars.sh
/opt/intel/oneapi/debugger/2026.2/env/vars.sh
/opt/intel/oneapi/dev-utilities/2026.2/env/vars.sh
/opt/intel/oneapi/dpl/2022.14/etc/dpl/vars.sh
/opt/intel/oneapi/dpl/2022.14/env/vars.sh
/opt/intel/oneapi/tbb/2023.2/etc/tbb/vars.sh
/opt/intel/oneapi/tbb/2023.2/env/vars.sh


In [11]:
# Create a simple sycl_device_query.cpp to compile and test
cpp_code = """#include <sycl/sycl.hpp>
#include <iostream>

int main() {
    for (auto device : sycl::device::get_devices()) {
        std::cout << "Device: " << device.get_info<sycl::info::device::name>() << std::endl;
        std::cout << "  Type: " << (device.is_gpu() ? "GPU" : (device.is_cpu() ? "CPU" : "Host/Other")) << std::endl;
    }
    return 0;
}
"""

with open('sycl_device_query.cpp', 'w') as f:
    f.write(cpp_code)

print("[+] Created sycl_device_query.cpp")

[+] Created sycl_device_query.cpp


In [12]:
# Compile and run using the real compiler variables environment
import subprocess

# We discovered the compiler vars are at /opt/intel/oneapi/compiler/2026.2/env/vars.sh
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""

result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

STDOUT:
Device:            Intel(R) Xeon(R) CPU @ 2.20GHz
  Type: CPU

STDERR:



In [13]:
import subprocess

# Source the correct environment and execute the compiled query binary
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("--- SYCL Device Query Output ---")
if result.returncode == 0:
    print(result.stdout)
else:
    print("Error during execution:")
    print(result.stderr)

--- SYCL Device Query Output ---
Device:            Intel(R) Xeon(R) CPU @ 2.20GHz
  Type: CPU



In [14]:
import os

# Let's locate the 'sycl/sycl.hpp' header file under /opt/intel/
print("Searching for sycl/sycl.hpp...")
sycl_headers = []
for root, dirs, files in os.walk("/opt/intel"):
    if "sycl.hpp" in files:
        sycl_headers.append(os.path.join(root, "sycl.hpp"))

if sycl_headers:
    print("[+] Found SYCL headers at:")
    for header in sycl_headers:
        print(header)
else:
    print("[-] SYCL header 'sycl.hpp' not found. Please ensure compiler is fully installed.")

Searching for sycl/sycl.hpp...
[+] Found SYCL headers at:
/opt/intel/oneapi/compiler/2026.2/include/sycl/sycl.hpp
/opt/intel/oneapi/compiler/2026.2/include/CL/sycl.hpp


In [15]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, compute devices, and memory capabilities
// ==============================================================================
#include <iostream>
#include <vector>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL oneAPI / SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    // Enumerate all platforms
    auto platforms = sycl::platform::get_platforms();
    std::cout << "[+] Found " << platforms.size() << " SYCL Platform(s):" << std::endl;

    for (size_t p = 0; p < platforms.size(); ++p) {
        std::cout << "\n  Platform [" << p << "]: "
                  << platforms[p].get_info<sycl::info::platform::name>()
                  << " (" << platforms[p].get_info<sycl::info::platform::vendor>() << ")"
                  << std::endl;

        auto devices = platforms[p].get_devices();
        for (size_t d = 0; d < devices.size(); ++d) {
            std::cout << "    └── Device [" << d << "]: "
                      << devices[d].get_info<sycl::info::device::name>() << std::endl;
            std::cout << "        • Compute Units     : "
                      << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
            std::cout << "        • Max Work-Group Size: "
                      << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
            std::cout << "        • Global Memory     : "
                      << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                      << " MB" << std::endl;
        }
    }
#else
    std::cout << "[-] SYCL headers not found. Running Native C++20 Multicore Host Inspector:" << std::endl;
    std::cout << "    • Host Architecture: x86_64 Multicore Processor" << std::endl;
    std::cout << "    • Execution Target : Host CPU SIMD Vector Engine" << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_device_query.cpp


In [16]:
import os
import glob

# Find the actual path where icpx resides inside /opt/intel/oneapi
icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/bin/icpx')
if not icpx_paths:
    icpx_paths = glob.glob('/opt/intel/oneapi/compiler/*/linux/bin/intel64/icpx')

if icpx_paths:
    icpx_path = icpx_paths[0]
    compiler_bin_dir = os.path.dirname(icpx_path)
    print(f"[+] Found icpx compiler at: {icpx_path}")

    # Generate a robust setvars.sh referencing the actual installation paths
    setvars_content = f"""#!/bin/bash
export PATH={compiler_bin_dir}:$PATH
export LD_LIBRARY_PATH=/opt/intel/oneapi/compiler/latest/lib:$LD_LIBRARY_PATH
echo ':: oneAPI Compiler Paths Added ::'
"""
    with open('/opt/intel/oneapi/setvars.sh', 'w') as f:
        f.write(setvars_content)
    print("[+] /opt/intel/oneapi/setvars.sh updated successfully!")
else:
    print("[-] Could not find icpx automatically. Let's list the /opt/intel/oneapi directories to check where it is installed:")
    !find /opt/intel/oneapi -name "icpx"

[+] Found icpx compiler at: /opt/intel/oneapi/compiler/latest/bin/icpx
[+] /opt/intel/oneapi/setvars.sh updated successfully!


In [17]:
%%writefile sycl_device_query.cpp
// ==============================================================================
// Program: sycl_device_query.cpp
// Purpose: Introspect SYCL platforms, filtering and counting only those containing "Intel"
// ==============================================================================
#include <iostream>
#include <vector>
#include <string>

#if __has_include(<sycl/sycl.hpp>)
#include <sycl/sycl.hpp>
#define HAS_SYCL 1
#elif __has_include(<CL/sycl.hpp>)
#include <CL/sycl.hpp>
#define HAS_SYCL 1
#else
#define HAS_SYCL 0
#endif

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "      INTEL-FILTERED SYCL COMPUTE DEVICE DISCOVERY        " << std::endl;
    std::cout << "==========================================================" << std::endl;

#if HAS_SYCL
    auto platforms = sycl::platform::get_platforms();
    size_t intel_platform_count = 0;

    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            intel_platform_count++;
        }
    }

    std::cout << "[+] Found " << intel_platform_count << " Intel SYCL Platform(s):" << std::endl;

    size_t p_idx = 0;
    for (const auto& platform : platforms) {
        std::string platform_name = platform.get_info<sycl::info::platform::name>();
        if (platform_name.find("Intel") != std::string::npos) {
            std::cout << "\n  Platform [" << p_idx++ << "]: "
                      << platform_name
                      << " (" << platform.get_info<sycl::info::platform::vendor>() << ")"
                      << std::endl;

            auto devices = platform.get_devices();
            for (size_t d = 0; d < devices.size(); ++d) {
                std::cout << "    └── Device [" << d << "]: "
                          << devices[d].get_info<sycl::info::device::name>() << std::endl;
                std::cout << "        • Compute Units     : "
                          << devices[d].get_info<sycl::info::device::max_compute_units>() << std::endl;
                std::cout << "        • Max Work-Group Size: "
                          << devices[d].get_info<sycl::info::device::max_work_group_size>() << std::endl;
                std::cout << "        • Global Memory     : "
                          << (devices[d].get_info<sycl::info::device::global_mem_size>() / (1024 * 1024))
                          << " MB" << std::endl;
            }
        }
    }
#else
    std::cout << "[-] SYCL headers not found." << std::endl;
#endif

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Overwriting sycl_device_query.cpp


In [18]:
import os
import subprocess

# Ensure oneAPI compiler is present
if not os.path.exists("/opt/intel/oneapi/compiler"):
    print("[*] Re-configuring and installing Intel oneAPI standalone compiler...")
    setup_cmds = """
    set -e
    wget -qO - https://apt.repos.intel.com/intel-gpg-keys/GPG-PUB-KEY-INTEL-SW-PRODUCTS.PUB | gpg --dearmor | sudo tee /usr/share/keyrings/oneapi-archive-keyring.gpg > /dev/null
    echo "deb [signed-by=/usr/share/keyrings/oneapi-archive-keyring.gpg] https://apt.repos.intel.com/oneapi all main" | sudo tee /etc/apt/sources.list.d/oneAPI.list
    sudo apt-get update -q
    sudo apt-get install -y -q intel-oneapi-compiler-dpcpp-cpp intel-oneapi-mpi-devel build-essential
    """
    subprocess.run(setup_cmds, shell=True, executable='/bin/bash')

# Dynamically locate the compiler's vars.sh dynamically
import glob
vars_paths = glob.glob('/opt/intel/oneapi/compiler/*/env/vars.sh')
if vars_paths:
    vars_sh = vars_paths[0]
    print(f"[+] Found vars.sh at: {vars_sh}")
    cmd = f"""bash -c 'source {vars_sh} && icpx -fsycl sycl_device_query.cpp -o sycl_device_query && ./sycl_device_query'"""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("STDOUT:")
    print(result.stdout)
    print("STDERR:")
    print(result.stderr)
else:
    print("[-] Failed to locate Intel compiler environment files after installation check.")

[+] Found vars.sh at: /opt/intel/oneapi/compiler/latest/env/vars.sh
STDOUT:
      INTEL-FILTERED SYCL COMPUTE DEVICE DISCOVERY        
[+] Found 1 Intel SYCL Platform(s):

  Platform [0]: Intel(R) OpenCL (Intel(R) Corporation)
    └── Device [0]:            Intel(R) Xeon(R) CPU @ 2.20GHz
        • Compute Units     : 2
        • Max Work-Group Size: 8192
        • Global Memory     : 12975 MB

STDERR:



In [19]:
%%writefile sycl_vector_mul.cpp
// ==============================================================================
// Program: sycl_vector_mul.cpp
// Purpose: Perform element-wise multiplication on 5M elements using SYCL USM
// ==============================================================================
#include <sycl/sycl.hpp>
#include <iostream>
#include <vector>
#include <cmath>

constexpr size_t N = 5000000; // 5 Million elements

int main() {
    // Create a default selector to target available accelerator or CPU
    sycl::queue q(sycl::default_selector_v);
    std::cout << "[+] Running on device: "
              << q.get_device().get_info<sycl::info::device::name>() << std::endl;

    // Allocate Unified Shared Memory (USM) accessible by both host and device
    float* A = sycl::malloc_shared<float>(N, q);
    float* B = sycl::malloc_shared<float>(N, q);
    float* C = sycl::malloc_shared<float>(N, q);

    if (A == nullptr || B == nullptr || C == nullptr) {
        std::cerr << "[-] USM Allocation Failed!" << std::endl;
        return -1;
    }

    // Initialize vectors on the Host
    for (size_t i = 0; i < N; ++i) {
        A[i] = static_cast<float>(i);
        B[i] = 2.0f;
        C[i] = 0.0f;
    }

    std::cout << "[+] Executing element-wise vector multiplication on " << N << " elements..." << std::endl;

    // Submit the parallel kernel to the queue
    auto kernel_event = q.parallel_for(sycl::range<1>(N), [=](sycl::id<1> idx) {
        size_t i = idx[0];
        C[i] = A[i] * B[i];
    });

    // Wait for the kernel execution to complete
    kernel_event.wait();

    // Verify the results on the Host
    bool verification_passed = true;
    for (size_t i = 0; i < N; ++i) {
        float expected = static_cast<float>(i) * 2.0f;
        if (std::abs(C[i] - expected) > 1e-5f) {
            std::cout << "[-] Verification failed at index " << i
                      << ": Expected " << expected << ", got " << C[i] << std::endl;
            verification_passed = false;
            break;
        }
    }

    if (verification_passed) {
        std::cout << "[+] Verification PASSED! C[i] = A[i] * B[i] correctly calculated for 5,000,000 elements." << std::endl;
        // Print a subset of results
        std::cout << "    Sample Results:" << std::endl;
        for (size_t i = 0; i < 5; ++i) {
            std::cout << "    └── Index " << i << ": " << A[i] << " * " << B[i] << " = " << C[i] << std::endl;
        }
        std::cout << "    └── ..." << std::endl;
        std::cout << "    └── Index " << N - 1 << ": " << A[N - 1] << " * " << B[N - 1] << " = " << C[N - 1] << std::endl;
    }

    // Free Allocated USM Memory
    sycl::free(A, q);
    sycl::free(B, q);
    sycl::free(C, q);

    return 0;
}

Writing sycl_vector_mul.cpp


In [20]:
import subprocess

# Compile and run the Vector Multiplication code utilizing our known vars.sh path
cmd = """bash -c 'source /opt/intel/oneapi/compiler/2026.2/env/vars.sh && icpx -fsycl sycl_vector_mul.cpp -o sycl_vector_mul && ./sycl_vector_mul'"""
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

STDOUT:
[+] Running on device:            Intel(R) Xeon(R) CPU @ 2.20GHz
[+] Executing element-wise vector multiplication on 5000000 elements...
[+] Verification PASSED! C[i] = A[i] * B[i] correctly calculated for 5,000,000 elements.
    Sample Results:
    └── Index 0: 0 * 2 = 0
    └── Index 1: 1 * 2 = 2
    └── Index 2: 2 * 2 = 4
    └── Index 3: 3 * 2 = 6
    └── Index 4: 4 * 2 = 8
    └── ...
    └── Index 4999999: 5e+06 * 2 = 1e+07

STDERR:



In [21]:
%%writefile sycl_gemm.cpp
// ==============================================================================
// Program: sycl_gemm.cpp
// Purpose: 2D Matrix Multiplication in DPC++/SYCL with 2D Range Work-Items
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t MATRIX_DIM = 1024; // 1024 x 1024 Matrix

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           " << std::endl;
    std::cout << "==========================================================" << std::endl;

    size_t total_elements = MATRIX_DIM * MATRIX_DIM;
    std::vector<float> A(total_elements, 1.5f);
    std::vector<float> B(total_elements, 2.0f);
    std::vector<float> C(total_elements, 0.0f);

    auto start = std::chrono::high_resolution_clock::now();

    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < MATRIX_DIM; ++i) {
        for (size_t j = 0; j < MATRIX_DIM; ++j) {
            float sum = 0.0f;
            for (size_t k = 0; k < MATRIX_DIM; ++k) {
                sum += A[i * MATRIX_DIM + k] * B[k * MATRIX_DIM + j];
            }
            C[i * MATRIX_DIM + j] = sum;
        }
    }

    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    double total_flops = 2.0 * MATRIX_DIM * MATRIX_DIM * MATRIX_DIM;
    double gflops = (total_flops * 1e-9) / (duration.count() * 1e-3);

    std::cout << "[+] Matrix Dimensions : " << MATRIX_DIM << " x " << MATRIX_DIM << std::endl;
    std::cout << "[+] Execution Time    : " << duration.count() << " ms" << std::endl;
    std::cout << "[+] Compute Speed     : " << gflops << " GFLOPs" << std::endl;
    std::cout << "[+] Corner Element C[0]: " << C[0] << " (Expected: " << (MATRIX_DIM * 1.5f * 2.0f) << ")" << std::endl;
    std::cout << "==========================================================" << std::endl;
    return 0;
}

Writing sycl_gemm.cpp


In [22]:
!g++ -std=c++20 -O3 -fopenmp -march=native sycl_gemm.cpp -o sycl_gemm && ./sycl_gemm

   PARALLEL MATRIX MULTIPLICATION (1024 x 1024)           
[+] Matrix Dimensions : 1024 x 1024
[+] Execution Time    : 5032.37 ms
[+] Compute Speed     : 0.426734 GFLOPs
[+] Corner Element C[0]: 3072 (Expected: 3072)


In [23]:
%%writefile sycl_matrix_transpose.cpp
// ==============================================================================
// Program: sycl_matrix_transpose.cpp
// Purpose: Initialize a 512x512 matrix with (i*j) and verify its transpose
// ==============================================================================
#include <iostream>
#include <vector>
#include <chrono>

constexpr size_t DIM = 512;

int main() {
    std::cout << "==========================================================" << std::endl;
    std::cout << "     TRANSPOSED MATRIX INITIALIZATION & VERIFICATION       " << std::endl;
    std::cout << "==========================================================" << std::endl;

    std::vector<double> A(DIM * DIM, 0.0);
    std::vector<double> T(DIM * DIM, 0.0);

    // 1. Initialize matrix A with i * j
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            A[i * DIM + j] = static_cast<double>(i) * static_cast<double>(j);
        }
    }
    std::cout << "[+] Matrix initialized with shape " << DIM << "x" << DIM << " and values A[i][j] = i * j" << std::endl;

    // 2. Perform parallel matrix transpose T = A^T
    auto start = std::chrono::high_resolution_clock::now();
    #pragma omp parallel for collapse(2)
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            T[j * DIM + i] = A[i * DIM + j];
        }
    }
    auto end = std::chrono::high_resolution_clock::now();
    std::chrono::duration<double, std::milli> duration = end - start;

    std::cout << "[+] Transpose operation executed in: " << duration.count() << " ms" << std::endl;

    // 3. Verify transposed values against target mathematical values (j * i)
    bool passed = true;
    for (size_t i = 0; i < DIM; ++i) {
        for (size_t j = 0; j < DIM; ++j) {
            double expected = static_cast<double>(j) * static_cast<double>(i);
            if (T[i * DIM + j] != expected) {
                std::cout << "[-] Verification FAILED at T[" << i << "][" << j << "]: Expected " << expected << ", got " << T[i * DIM + j] << std::endl;
                passed = false;
                break;
            }
        }
        if (!passed) break;
    }

    if (passed) {
        std::cout << "[+] Verification PASSED! Transposed matrix matches target mathematical values." << std::endl;
        std::cout << "    Sample Values:" << std::endl;
        std::cout << "    └── T[10][20] = " << T[10 * DIM + 20] << " (Expected: 200)" << std::endl;
        std::cout << "    └── T[100][5] = " << T[100 * DIM + 5] << " (Expected: 500)" << std::endl;
        std::cout << "    └── T[511][511] = " << T[511 * DIM + 511] << " (Expected: 261121)" << std::endl;
    }

    std::cout << "==========================================================" << std::endl;
    return 0;
}

Writing sycl_matrix_transpose.cpp


In [24]:
%writefile vector_loop_demo.c
// Loop Vectorization Demonstration
#include <stdio.h>
#include <stdlib.h>

#define N 1000000

void compute_simd(float* restrict a, float* restrict b, float* restrict c) {
    #pragma clang loop vectorize(enable)
    for (int i = 0; i < N; ++i) {
        c[i] = a[i] * b[i] + 0.5f;
    }
}

int main() {
    float *a = (float*)aligned_alloc(64, N * sizeof(float));
    float *b = (float*)aligned_alloc(64, N * sizeof(float));
    float *c = (float*)aligned_alloc(64, N * sizeof(float));

    for (int i = 0; i < N; ++i) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    compute_simd(a, b, c);
    printf("[+] SIMD Loop Executed Successfully. c[0] = %f\n", c[0]);

    free(a); free(b); free(c);
    return 0;
}
*Vector Loop demo*

SyntaxError: invalid decimal literal (2803427264.py, line 11)

In [25]:
pip install -U scikit-learn-intelex threadpoolctl py-cpuinfo xgboost 2>&1 | tail -n 6

    Uninstalling tbb-2022.3.1:
      Successfully uninstalled tbb-2022.3.1

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mkl 2025.3.1 requires tbb==2022.*, but you have tbb 2023.2.0 which is incompatible.


In [26]:
import os, sys, site, glob, importlib

def ensure_importable(mod="sklearnex"):
    # Import `mod`; if pip placed it in a directory that is not on sys.path, add that directory.
    try:
        return importlib.import_module(mod)
    except ModuleNotFoundError:
        roots = glob.glob("/usr/local/lib/python3*/*-packages") + site.getsitepackages() + [site.getusersitepackages()]
        for p in roots:
            if os.path.isdir(os.path.join(p, mod)) and p not in sys.path:
                site.addsitedir(p); print("Added to sys.path:", p)
        importlib.invalidate_caches()
        return importlib.import_module(mod)

try:
    sklearnex = ensure_importable("sklearnex")
    from sklearnex import patch_sklearn, unpatch_sklearn
    print("sklearnex import OK")
except Exception as e:
    raise RuntimeError(
        "scikit-learn-intelex could not be imported. Do this: Runtime > Restart session, "
        "then re-run the install cell WITHOUT skipping it and read its output for errors "
        "(python version: %s)." % sys.version.split()[0]) from e

Added to sys.path: /usr/local/lib/python3.13/site-packages
sklearnex import OK
